In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [2]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [3]:
os.chdir("/home/lishaogang/NEU2026-Jan")
adata_NK = sc.read_h5ad('adata_NK_raw.h5ad')

In [13]:
# 2. 重新过滤基因（只保留在当前 NK 细胞子集中有表达的基因）
sc.pp.filter_genes(adata_NK, min_cells=3)

# 3. 运行 HVG
print("\n--- Finding HVGs within the T-cell subset ---")
# 方案 A: 依然使用 batch_key（如果样本量大且细胞多，推荐）

sc.pp.highly_variable_genes(adata_NK, batch_key='dataset', n_top_genes=2000, flavor='seurat')


# 4. 提取 HVG 并后续处理
adata_NK_hvg = adata_NK[:, adata_NK.var.highly_variable].copy()
sc.pp.scale(adata_NK_hvg, zero_center=False)
sc.tl.pca(adata_NK_hvg, n_comps=50, random_state=seed)

print("✅ PCA 计算完成！")

In [14]:
# --- Drawing a Scree Plot ---
print("Drawing a scree plot to assist in determining the selection of n_pcs...")

# 1. Modify Matplotlib's global default figure size before plotting
#    This will affect all figures created afterwards
plt.rcParams['figure.figsize'] = (8, 6)  # Set to 8x6 inches

# 2. Now directly call scanpy's plotting function without any additional parameters
#    When it internally creates the figure, it will follow the global size set above
sc.pl.pca_variance_ratio(adata_NK_hvg, log=True, n_pcs=50)
plt.show()

# 3. Restore Matplotlib's default settings
#    This is a good practice to prevent this size setting from affecting your subsequent plots
plt.rcParams.update(plt.rcParamsDefault)

In [17]:
# --- 5. 运行 Harmony (校正批次效应) ---
import scanpy.external as sce

print("\n--- 正在为 NK 亚群运行 Harmony ---")
sce.pp.harmony_integrate(
    adata_NK_hvg, 
    key='dataset', 
    basis='X_pca', 
    adjusted_basis='X_pca_harmony'
)

# --- 6. 映射结果并聚类 ---
adata_NK.obsm['X_pca_harmony'] = adata_NK_hvg.obsm['X_pca_harmony']

# 计算邻居和 UMAP (建议将邻居数调小一点，比如 10-15，因为 NK 子集通常比较小)
sc.pp.neighbors(adata_NK, n_neighbors=15,n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_NK)




In [19]:
sc.tl.leiden(adata_NK, resolution=0.2, key_added='leiden_NK')

In [20]:
# --- 7. 可视化 ---
sc.pl.umap(adata_NK, color=['leiden_NK', 'dataset', 'Disease'], wspace=0.4)

In [22]:

NK_gene_list = ['B3GAT1', 'CX3CR1', 'FCGR3A', 'GATA3', 'GZMK', 'IL7R', 'KLRC1', 'KIT', 'MKI67', 'NCAM1', 'STMN1', 'TYMS', 'XCL1', 'ZEB2', 'CD160']


In [27]:
# --- Plot Dot Plot for Cell Type Annotation ---
# Dot plot is an excellent way to show the expression of specific genes in each cluster.
# The size of the dot represents the proportion of cells expressing the gene, and the color intensity represents the average expression level.
# **`use_raw=True`** is a key parameter that ensures we use the normalized and log-transformed raw expression values,
# rather than scaled values, which is more accurate for biological interpretation.
print("Generating dot plot...")
dotplot = sc.pl.dotplot(
    adata_NK,
    NK_gene_list,
    groupby="leiden_NK",
    use_raw=True,         # <-- Key! Use data from .raw
    #dendrogram=True,      # Perform hierarchical clustering on clusters, similar clusters will be grouped together
    standard_scale='var', # Normalize by gene to make genes with different expression levels comparable
    cmap='Reds',         # Use blue color scheme
    figsize=(18, 10),     # Set figure size
    return_fig=True,      # Return figure object for further modification
    show=False            # Do not display the figure immediately
)

# --- Customize and Beautify Dot Plot ---
print("Beautifying dot plot...")
dotplot.make_figure() # Generate the figure
main_ax = dotplot.ax_dict['mainplot_ax'] # Get the axes object of the main plot

# Add grid lines to make the figure easier to read
# Add horizontal lines (between genes)
main_ax.hlines(y=np.arange(len(NK_gene_list)) - 0.5, xmin=-0.5, xmax=len(adata_NK.obs['leiden_NK'].cat.categories) - 0.5, color='lightgray', lw=0.5, ls='--')
# Add vertical lines (between clusters)
main_ax.vlines(x=np.arange(len(adata_NK.obs['leiden_NK'].cat.categories)) - 0.5, ymin=-0.5, ymax=len(NK_gene_list) - 0.5, color='lightgray', lw=0.5, ls='--')

plt.show()


print("\n--- Exporting Raw Data of Dot Plot ---")
mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     print("\n--- Mean Expression Matrix (Color) ---")
#     #print(mean_expression_df)
#     print("\n--- Cell Expression Fraction Matrix (Dot Size) ---")
#     #print(fraction_df)



# 2. Call the function to generate a summary table
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

# 3. Convert the resulting DataFrame to a tab-separated string and print it
tsv_output = final_summary_df.to_csv(sep='\t')

#print("--- Gene Expression Level Summary Table (Tab-Separated) ---")
print(tsv_output)

In [29]:
summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)

In [36]:
annotation_dict = {
    "0": "NK dim",
    "1": "NK bright",
    "2": "NK dim",
    "3": "Transit. NK",
    "4": "NK dim",
    "5": "NK dim",
    "6": "ILC2"
}

In [37]:
# 将注释映射到 adata 对象
adata_NK.obs['cell_subtype'] = adata_NK.obs['leiden_NK'].map(annotation_dict).astype('category')

# 可视化检查
import scanpy as sc
sc.pl.umap(adata_NK, color='cell_subtype', title='ILC Subsets')

In [38]:
#保存
adata_NK.write_h5ad("adata_NK_final.h5ad")

In [40]:
dotplot = sc.pl.dotplot(
    adata_NK,
    NK_gene_list,
    groupby="cell_subtype",
    use_raw=True,         # <-- Key! Use data from .raw
    #dendrogram=True,      # Perform hierarchical clustering on clusters, similar clusters will be grouped together
    standard_scale='var', # Normalize by gene to make genes with different expression levels comparable
    cmap='Reds',         # Use blue color scheme
    figsize=(18, 10),     # Set figure size
    return_fig=True,      # Return figure object for further modification
    show=False            # Do not display the figure immediately
)

# --- Customize and Beautify Dot Plot ---
print("Beautifying dot plot...")
dotplot.make_figure() # Generate the figure
main_ax = dotplot.ax_dict['mainplot_ax'] # Get the axes object of the main plot

# Add grid lines to make the figure easier to read
# Add horizontal lines (between genes)
main_ax.hlines(y=np.arange(len(NK_gene_list)) - 0.5, xmin=-0.5, xmax=len(adata_NK.obs['leiden_NK'].cat.categories) - 0.5, color='lightgray', lw=0.5, ls='--')
# Add vertical lines (between clusters)
main_ax.vlines(x=np.arange(len(adata_NK.obs['leiden_NK'].cat.categories)) - 0.5, ymin=-0.5, ymax=len(NK_gene_list) - 0.5, color='lightgray', lw=0.5, ls='--')

plt.show()


In [56]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

# ========================
# 1. 第一步：定义 NK/ILC 亚群特异性 Marker 字典
# ========================
# 按你想要的阶梯顺序排列：ILC2 -> NK bright -> Transit. NK -> NK dim
# 这样 Y 轴和 X 轴会自动对齐形成阶梯
subtype_markers = {
    "ILC2": ["IL7R", "KIT", "GATA3"],
    "NK bright": ["XCL1", "KLRC1", "GZMK","NCAM1",],
    "Transit. NK": [ "FCGR3A", "GZMB", ],
    "NK dim": ["B3GAT1", "CX3CR1", "ZEB2", "PRF1" ,'CD160','ZEB2']
}

# ========================
# 2. 第二步：有效基因检测与自动分组
# ========================
# 检查是在 .X 还是 .raw 中寻找基因
var_names_source = adata_NK.raw.var_names if (adata_NK.raw is not None) else adata_NK.var_names

valid_markers = {}
for subtype, genes in subtype_markers.items():
    # 只保留数据集中存在的基因
    existing = [g for g in genes if g in var_names_source]
    if existing:
        valid_markers[subtype] = existing

print(f"有效分组基因：{valid_markers}")

# ========================
# 3. 第三步：细胞类型排序（实现对角线的核心）
# ========================
# 确保 Y 轴顺序与字典顺序一致：ILC2, NK bright, Transit. NK, NK dim
target_order = [s for s in subtype_markers.keys() if s in adata_NK.obs['cell_subtype'].cat.categories]
adata_NK.obs['cell_subtype'] = adata_NK.obs['cell_subtype'].cat.reorder_categories(target_order)




In [58]:

# ========================
# 4. 第四步：绘制 Dotplot
# ========================
# 1. 动态计算尺寸
n_genes_total = sum(len(v) for v in valid_markers.values())
fig_width = max(12, n_genes_total * 0.8) # 更加紧凑的比例

# 2. 绘图
# 使用字典作为 var_names，Scanpy 会自动在图顶部生成分组标签
dp = sc.pl.dotplot(
    adata_NK,
    var_names=valid_markers,
    groupby='cell_subtype',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(fig_width, 5),
    show=False,
    return_fig=True
)

# 3. 获取 figure 对象进行最后微调
fig = dp.make_figure()

# 4. 美化：手动调整底部标签旋转（解决警告并防止截断）
# Scanpy 的索引通常在 'mainplot_ax'
ax = dp.ax_dict['mainplot_ax']
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# 5. 保存并展示
# 使用 bbox_inches='tight' 自动解决边距问题，不再需要复杂的 subplots_adjust
plt.savefig(
    'NK_staircase_dotplot.png', 
    dpi=300, 
    bbox_inches='tight',
    facecolor='white'
)
plt.show()